In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline

In [17]:
db = SQLDatabase.from_uri("sqlite:///chinook.db")
def get_schema(_):
    return db.get_table_info()

def run_query(query):
    print(f'Query being run : {query} \n\n')
    return db.run(query)


In [8]:
print(get_schema(None))


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Empl

In [1]:
import math
from langchain_community.utilities import SQLDatabase

# 1. Initialize your DB connection
db = SQLDatabase.from_uri("sqlite:///chinook.db")

# 2. Fetch all usable table names
table_names = db.get_usable_table_names()

print("--- Database Schema Size Analysis ---\n")

max_chars = 0
largest_table = ""

# 3. Loop through tables to check individual sizes
for table in table_names:
    table_info = db.get_table_info([table])
    char_length = len(table_info)
    
    print(f"🔹 Table: '{table:<15}' ➡️ Length: {char_length:,} characters")
    
    # Track the absolute largest table
    if char_length > max_chars:
        max_chars = char_length
        largest_table = table

print("\n-------------------------------------")
print(f"🏆 Largest Table: '{largest_table}'")
print(f"📊 Maximum Character Length: {max_chars:,} characters")
print("-------------------------------------")

# 4. Calculate recommended chunk size (Rounding up to nearest 100 + adding buffer)
safety_buffer = 300
recommended_chunk_size = math.ceil((max_chars + safety_buffer) / 100) * 100

print(f"\n💡 [RECOMMENDATION]")
print(f"Set your RecursiveCharacterTextSplitter settings to:")
print(f"👉 chunk_size    = {recommended_chunk_size}")
print(f"👉 chunk_overlap = 0")


C:\Users\gurpr\AppData\Local\Temp\ipykernel_36840\3753945115.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


--- Database Schema Size Analysis ---

🔹 Table: 'Album          ' ➡️ Length: 346 characters
🔹 Table: 'Artist         ' ➡️ Length: 183 characters
🔹 Table: 'Customer       ' ➡️ Length: 1,063 characters
🔹 Table: 'Employee       ' ➡️ Length: 1,198 characters
🔹 Table: 'Genre          ' ➡️ Length: 171 characters
🔹 Table: 'Invoice        ' ➡️ Length: 786 characters
🔹 Table: 'InvoiceLine    ' ➡️ Length: 478 characters
🔹 Table: 'MediaType      ' ➡️ Length: 244 characters
🔹 Table: 'Playlist       ' ➡️ Length: 192 characters
🔹 Table: 'PlaylistTrack  ' ➡️ Length: 338 characters
🔹 Table: 'Track          ' ➡️ Length: 934 characters

-------------------------------------
🏆 Largest Table: 'Employee'
📊 Maximum Character Length: 1,198 characters
-------------------------------------

💡 [RECOMMENDATION]
Set your RecursiveCharacterTextSplitter settings to:
👉 chunk_size    = 1500
👉 chunk_overlap = 0


THIS IS PHASE 1 IN WHICH I AM ASSUMING THAT THE CHUNK SIZE IS NOT VERY HUGE , TO HAMPER THE QUALITY OF EMBEDDINGS 

In [6]:
from langchain_community.utilities import SQLDatabase
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize your DB connection
db = SQLDatabase.from_uri("sqlite:///chinook.db")


# 2. Get the raw text schema for all tables
def get_schema(_=None):
    return db.get_table_info()


full_schema_text = get_schema()

# 3. Create a Document wrapper
doc = Document(page_content=full_schema_text, metadata={"source": "sql_schema"})

# 4. Split by the CREATE TABLE statement boundary
# This ensures each table definition stays cleanly together in one chunk
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1198,  # Adjust size based on how many sample rows your schema includes
    chunk_overlap=0,
    separators=["\n\nCREATE TABLE"],  # Standard boundary separating tables in LangChain's output
)

schema_chunks = text_splitter.split_documents([doc])

for i, chunk in enumerate(schema_chunks):
    print(f"----{i+1}--- \n {chunk.page_content}\n\n")




----1--- 
 CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


----2--- 
 CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRep

There was a problem in the recursive character text splitter, since the chunk size was kept 1198(size of the biggest schema) multiple small table schemas git combined into a single chunk, we need to rectify it in the fuure

In [5]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpointEmbeddings

load_dotenv()

hf_token = os.getenv("HF_TOKEN") 

embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2", 
    huggingfacehub_api_token=hf_token,
)

texts_to_embed = [doc.page_content for doc in schema_chunks]
vectors = embeddings.embed_documents(texts_to_embed)

print(f"Successfully generated {len(vectors)} vectors!")


Successfully generated 7 vectors!


In [19]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='my_chroma_db',
    collection_name='sample'
)

vector_store.add_documents(schema_chunks)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [40]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chat_history = []
with open('chat_history.txt', 'r') as f:
    chat_history = [HumanMessage(content=line.strip()) for line in f if line.strip()]

print(chat_history)

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_openai import ChatOpenAI

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

def get_llm():
    endpoint = HuggingFaceEndpoint(
        repo_id="Qwen/Qwen2.5-72B-Instruct",
        task="text-generation",
        temperature=0.1,
        max_new_tokens=512,
    )
    return ChatHuggingFace(llm=endpoint)

def retrieve_schema(question):
    documents = retriever.invoke(question)
    print(f"Retrieved {len(documents)} schema chunks for the question: '{question}'")
    return "\n\n".join(document.page_content for document in documents)

def write_sql_query(llm):
    
    template = """Based on the table schema below, write a SQL query that would answer the user's question:
    {schema}

    Question: {question}
    SQL Query:"""

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Given an input question, convert it to a SQL query. No pre-amble. "
                "Please do not return anything else apart from the SQL query, no prefix or suffix quotes, no sql keyword, nothing please."
            ),
            MessagesPlaceholder(variable_name='chat_history'),
            ("human", template),
        ]
    )
    
    return (
        RunnablePassthrough.assign(
            schema=lambda x: retrieve_schema(x["question"])
        )
        | prompt
        | llm
        | StrOutputParser()
    )

[]


In [41]:
def answer_user_query(query, llm, history):
    
    template = """Based on the table schema below, question, sql query, and sql response, write a natural language response:
    {schema}

    Question: {question}
    SQL Query: {query}
    SQL Response: {response}"""

    prompt_response = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Given an input question and SQL response, convert it to a natural language answer. No pre-amble.",
            ),
            ("human", template),
        ]
    )

    sql_chain = write_sql_query(llm)

    full_chain = (
        RunnablePassthrough.assign(
            schema=lambda x: retrieve_schema(x["question"])
        )
        | RunnablePassthrough.assign(
            query=lambda x: sql_chain.invoke(x)
        )
        | RunnablePassthrough.assign(
            response=lambda x: run_query(x["query"])
        )
        | prompt_response
        | llm
        | StrOutputParser()
    )

    return full_chain.invoke({
        "question": query,
        "chat_history": history
    })

In [ ]:
load_dotenv()
query = "give me some tracks by Audioslave"
    
    # 1. Pass the chat history into answer_user_query
    # 2. StrOutputParser returns a string directly, so print(response) is used instead of response.content
response = answer_user_query(query, llm=get_llm(), history=chat_history)
print(response)

Retrieved 4 schema chunks for the question: 'give me some tracks by Audioslave'
Retrieved 4 schema chunks for the question: 'give me some tracks by Audioslave'
Query being run : SELECT Track.Name FROM Track INNER JOIN Album ON Track.AlbumId = Album.AlbumId INNER JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Artist.Name = 'Audioslave' LIMIT 5 


